# BINF TP 11 : Genome wide association study

L'étude d'association pangénomique - Genome Wide Association study (GWAS) - est une méthode permettant de mettre en avant des mutations génétiques associées à une condition. Pour cela, on va recruter un ensemble de sujets possédant la condition que l’on souhaite étudier et un ensemble de sujets contrôle "sains". On va ensuite génotyper (ou séquencer) chaque sujet, c’est-à-dire déterminer les nucléotides présents à un ensembles de positions précises le long du génome. Ces positions sont connues pour posséder plusieurs nucléotides différents (des allèles) dans la population humaine qu'on appelle SNP (Single Nucleotide Polymorphism).

Dans ce TP nous allons chercher les gènes associés aux maladies cornariennes - ensemble de maladies où les artères vascularisant le cœur n’arrivent plus à apporter suffisamment de sang pour alimenter les muscles cardiaques. Nous allons utiliser les données de l’étude [PennCath](https://pmc.ncbi.nlm.nih.gov/articles/PMC3335297/). Cette étude comporte 1401 sujets pour lesquels on a génotypé 860473 positions.

Les données sont disponibles [ici](https://www.dropbox.com/scl/fo/tyt6sx74zevblzl4i5lpr/AAJt-WmbWCKmsPuRyzYPfP0?rlkey=yuklfyfy7b93yn2sow0uzuzub&dl=0). Notez que le fichier de génotypage complet avec 1401 patients fait > 2 go, pour faciliter le TP, nous utiliserons un fichier réduit avec seulement 100 patients.

## Exercice 1 : Chargement des données

Les données de l’étude sont divisées en plusieurs fichiers :


* penncath.bim : qui contient les informations sur les positions génotypées (SNPs), une position par ligne avec les colonnes suivantes :
  * Chromosome : pouvant contenir les valeurs 1 – 22, X ou Y.
  * Identifiant de la position dans le génome de référence.
  *	Distance génétique (ignorez cette colonne, elle n’est pas remplie dans le fichier).
  * Position sur le chromosome en paires de bases (on va aussi ignorer cette colonne).
  * Allèle mineure : allèle alternative dans la population (valeurs A,T,G,C ou -9 pour indiquer une absence de données).
  *	Allèle majeure : allèle la plus présente dans la population.

*	penncath.fam : qui contient les informations sur les sujets, une ligne par individu. Dans ce fichier, on s’intéresse uniquement à la première colonne qui contient l’identifiant de la famille.

*	penncath.csv : qui contient des informations cliniques sur les patients, en particulier s’il est atteint de la maladie coronarienne. Ce fichier contient une ligne par patient avec les colonnes suivantes :
   * Id du patient correspondant à la première colonne du fichier fam.
   * Maladie coronarienne : 1=OUI, 0=NON.
   * Les autres colonnes ne nous intéressent pas.

* genotype_small.csv : contient les donnes de génotypage, une colonne par patient, une ligne par position génotypée. Les valeurs sont : 0 : homozygote allèle majeur, 1 : hétérozygote, 2 : homozygote allèle mineure, 3 : génotype manquant.

Chargez les différents fichiers en faisant attention à bien maintenir l’ordre des lignes entre patients et génotypes et associez à chaque patient son statut malade ou sain.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

g_dat = []
with open("/content/penncath.bim", "r") as f:
  for line in f:
    g_dat.append(line.rstrip("\n").split(" "))

p_dat1 = []
with open("/content/penncath.fam", 'r') as f:
  for line in f:
    p_dat1.append(line.rstrip('\n').split(" "))

p_dat2 = []
with open("/content/penncath.csv", "r") as f:
  f.readline()
  for line in f:
    p_dat2.append(line.rstrip("\n").split(","))

p_dat = []
for p in p_dat1:
    for p2 in p_dat2:
        if p[0] == p2[0]:
            p_dat.append(int(p2[1]))
            break
p_dat = np.array(p_dat,dtype=int)

## Exercice 2 : contrôle qualité

Comme on manipule des données brutes, certaines vont être problématiques et on va vouloir les enlever pour éviter de biaiser les résultats. Dans cette exercice, nous allons détecter et filtrer les données problématiques.


1. Vérifions la distribution des données par chromosome. Affichez le nombre de SNP pour chaque chromosome sous la forme d'un barplot (l'axe x représente chaque chromosome et l'axe y le nombre de SNPs correspondant).

In [ ]:
chrs = np.array([int(e[0]) for e in g_dat])
plt.figure()
plt.bar([str(e) for e in range(1,23)], [sum(chrs == k) for k in range(1,23)])
plt.ylabel("Nombre de SNPs")
plt.xlabel("Chromosome")

2. Qu’observez-vous ?

On a de moins en moint de SNP plus on avance dans les chromosomes.
C'est normales car les chromosomes sont nommés en fonction de leurs tailles !

3. Pour obtenir une représentation plus claire, normalisons ces valeurs par le la taille du chromosome en nombre de nucléotides. Récupérez ces tailles depuis internet.

In [ ]:
sizes = np.array([247249719,242951149,199501827,191273063,180857866,170899992,
                  158821424,146274826,140273252,135374737,134452384,132349534,
                  114142980,106368585,100338915,88827254,78774742,76117153,
                  63811651,62435964,46944323,49691432,154913754,57772954])

chrs = np.array([int(e[0]) for e in g_dat])
plt.figure()
plt.bar([str(e) for e in range(1,23)], [sum(chrs == k) / sizes[k-1] for k in range(1,23)])
plt.ylabel("SNP par nucléotide")
plt.xlabel("Chromosome")

4. Qu'observez-vous maintenant ?

Maintenant les valeurs sont à peut près constantes, sauf pour le chromosome 18 qui a clairement moins de SNP que les autres.

5. Vérifions maintenant les valeurs des allèles. Affichez un barplot comptant la quantité de chaque nucléotide apparaissant dans les allèles mineures (-9 signifie une absence de données, affichez le dans le graphique, c'est important).


In [ ]:
bases = ["A", "T", "G", "C", "-9"]
min_as = np.array([bases.index(e[4]) for e in g_dat])

plt.figure()
plt.bar(bases, [sum(min_as == k) for k in range(5)])
plt.ylabel("Compte pour l'allèle mineure")

6. Faites de même avec les valeurs de l’allèle majeure.

In [ ]:
maj_as = np.array([bases.index(e[5]) for e in g_dat])
plt.figure()
plt.bar(bases, [sum(maj_as == k) for k in range(5)])
plt.ylabel("Compte pour l'allèle majeure")

7. Qu'observez-vous en comparant ces deux graphiques ?

Les fréquences des différents nucléotides sont a peut près égales. On a des nucléotides indéterminées pour l'allèle mineur uniquement.

8. Intéressons-nous maintenant aux données de génotypage. Dans ces données, une valeur NAN indique qu'il y a eu une erreur au niveau de l'acquisition des données. Affichez l'histogramme du nombre d'erreurs de génotypage par patient (nombre de NAN ou valeurs 4).

In [ ]:
gen = np.genfromtxt("/content/genotypes_small.csv", delimiter=",")

9. Affichez l'histogramme du nombre d'erreurs par position de SNP (nombre de NAN ou valeurs 4).

In [ ]:
plt.figure()
plt.hist(np.sum(np.isnan(gen), axis=1), bins=51)
plt.xlabel("Nombre d'erreurs")
plt.ylabel("Count")

## Exercice 3 : Association

Pour chaque SNP on va calculer sa probabilité d'être sur ou sous-représenté dans la population malade comparé à la population contrôle. Il existe plusieurs méthodes statistiques plus ou moins sophistiquées pour faire celà, nous allons voir la méthode la plus simple utilisant des tableaux de comptes d'allèles.

1. Pour chaque génotype $k$, construisez le tableau $2 \times 2$ suivant :
$$O=\begin{array}{|l|ll|}
\hline
& Allèle\ mineure & Allèle\ majeure\\\hline
sain & a & b\\
malade & c & d\\
\hline
\end{array}$$
où $a,b$ sont les nombres de patients sain possédant l'allèle mineure ou majeure (respectivement) et $c,d$ les nombres de patients malades possédant l'alèlle mineure ou majeure (respectivement), pour le génotype $k$.

A partir de ce tableau,calculez une p-value en effectuant un test du $\chi^2$ sur la table $O$ :

$$
z_{ij} = \frac{(o_{ij}-e_{ij})^2}{e_{ij}},
$$
où $o_{ij}$ est la valeure de la case $i,j$ de la table $O$ (la valeure observée) et $e$ la valeur attendue selon l'hypothèse nulle calculée par:
$$e_{ij} = \frac{\left(\sum\limits_{k=1}^{2}o_{k,j}\right)\left(\sum\limits_{k=1}^{2}o_{i,k}\right)}{\sum\limits_{k=1}^{2}\sum\limits_{l=1}^{2} o_{kl}}.$$

Finalement $z = z_{11} + z_{12} + z_{21} + z_{22}$ doit suivre une loi du $\chi^2$ à 1 degré de liberté. La p-value que le génotype mutant du SNP $k$ ait un effet sur la maladie est donnée par la probabilité associée à $z$.

Vous pouvez utilisez la fonction chi2_contingency de scipy.stats.

In [ ]:
from scipy.stats import chi2_contingency

p_dat = p_dat[:100]
ps = np.ones(gen.shape[1])
tab = np.zeros((2,2))
for k in range(gen.shape[1]):
    tab[0,0] = sum(gen[p_dat==0,k])
    tab[0,1] = sum(2 - gen[p_dat==0,k])
    tab[1,0] = sum(gen[p_dat==1,k])
    tab[1,1] = sum(2 - gen[p_dat==1,k])
    if any(sum(tab) == 0):
        continue
    ps[k] = chi2_contingency(tab)[1]

2. Affichez les résultat sous forme de graphique de Manhattan. Affichez toutes les paires (i, -log10(pval[i])) avec un marqueur « . ».

In [ ]:
plt.figure()
plt.plot(range(len(ps)), -np.log10(ps), '.')
plt.plot([0, len(ps)], [5, 5], 'r')
plt.ylabel("-log10(p)")
plt.xlabel("SNP")

3. Finalement, affichez les identifiants de tous les SNPs significativement plus présents chez les patients malades que sain. C'est à dire, les SNP avec
$$-log10(pval) > 5$$

In [ ]:
for k in range(ps.shape[0]):
    if -np.log10(ps[k]) > 5:
        print(g_dat[k][1], -np.log10(ps[k]))